# 11. Customer Support Handoffs — 상태 전환형 멀티에이전트

## 학습 목표

- handoff가 subagent delegation과 어떻게 다른지 구분합니다.
- 고객지원 상태 머신을 deterministic하게 설계합니다.
- escalation과 approval 조건을 명시합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class SupportState(TypedDict):
    message: str
    owner: str
    resolution: str

## 11.1 Handoff 기준

handoff는 “누가 다음 응답의 주체인가”가 바뀌는 패턴입니다.

In [ ]:
def triage(state: SupportState) -> dict:
    msg = state["message"].lower()
    if "refund" in msg or "환불" in msg:
        return {"owner": "billing"}
    if "error" in msg or "오류" in msg:
        return {"owner": "technical"}
    return {"owner": "general"}

In [ ]:
def resolve(state: SupportState) -> dict:
    templates = {
        "billing": "환불 정책을 확인하고 승인 요청을 준비합니다.",
        "technical": "오류 재현 정보와 로그를 요청합니다.",
        "general": "기본 안내를 제공합니다.",
    }
    return {"resolution": templates[state["owner"]]}

## 11.2 그래프로 표현하기

LangGraph를 쓰면 handoff 기록이 state에 남고 테스트하기 쉽습니다.

In [ ]:
builder = StateGraph(SupportState)
builder.add_node("triage", triage)
builder.add_node("resolve", resolve)
builder.add_edge(START, "triage")
builder.add_edge("triage", "resolve")
builder.add_edge("resolve", END)
graph = builder.compile()

In [ ]:
result = graph.invoke({
    "message": "구독 환불을 요청합니다.",
    "owner": "", "resolution": "",
})
result

## 11.3 승인 게이트

| 조건 | 처리 |
|---|---|
| 환불/결제 | human approval |
| 기술 오류 | 로그 요청 후 재현 |
| 일반 문의 | 자동 응답 가능 |

In [ ]:
def needs_approval(state: SupportState) -> bool:
    return state["owner"] == "billing"

print("approval required:", needs_approval(result))

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | handoff, state machine, approval gate |
| **핵심 개념** | handoff는 작업 위임이 아니라 대화 주체와 상태의 전환입니다. |
| **다음 단계** | `12_router_knowledge_base.ipynb` |

**참고 문서:**
- `docs/langchain/multi-agent/handoffs.md`
- `docs/langchain/multi-agent/handoffs-customer-support.md`
- `docs/langgraph/workflows-agents.md`